# NB1a — IV Cleaning & Cohort → `lucia_cells.parquet`

**Env:** `datasci`  
**Input:** `cell_data_dictionary_backup.json`  
**Output:** `data/processed/lucia_cells.parquet`  
**Schema:** `cell_name`(idx) · 11 IV cols · `wo_group` · `split` · `iv_keep`

Steps:
1. Load IV from JSON → DataFrame keyed by `canonical_name`; report key collisions
2. Coerce numeric; NaN counts per IV param
3. Hard exclusions → `iv_keep`
4. Count printout for candidate cuts (stop → user confirms)
5. Apply confirmed cuts
6. WO-grouped split (plain numpy — zero sklearn needed)
7. Save parquet; print kept/excluded + split sizes + describe()

In [69]:
# ── Config ──────────────────────────────────────────────────────────────────
# Purpose : single place for all thresholds; edit here, never scatter magic numbers
# Inputs  : nothing
# Outputs : constants used by all cells below

import json, os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, "/home/derk/DataScience/CAS_projects/LUCIA/notebooks_scripts")
from lucia_common import ROOT, PROCESSED, canonical_name

os.makedirs(PROCESSED, exist_ok=True)

IV_JSON     = f"{ROOT}/cell_data_dictionary_backup.json"
OUT_PARQUET = f"{PROCESSED}/lucia_cells.parquet"

# Hard exclusion thresholds (applied unconditionally)
HARD = dict(ff_lo=0.25, ff_hi=0.90, voc_lo=0.50, isc_lo=3.0)  # exclusion limits in absolute units (§2)

# Candidate lower-tail thresholds to count before user confirms
CAND_VOC_LO  = [0.68, 0.69, 0.70, 0.71]
CAND_ISC_LO  = [4.8, 4.9, 5.0, 5.1]
CAND_FF_LO   = [0.40, 0.45, 0.50]
CAND_PMAX_LO = [2.8, 2.9, 3.0]

# Candidate upper-tail thresholds to count
CAND_VOC_HI  = [0.747]   # firm per Rev 1.2
CAND_ISC_HI  = [5.80]
CAND_FF_HI   = [0.89]
CAND_PMAX_HI = [3.42]

# ── CONFIRMED CUTS (fill in after reviewing Cell 4 output, then re-run Cell 5) ──
CONFIRMED_VOC_LO  = None   # e.g. 0.70
CONFIRMED_ISC_LO  = None   # e.g. 4.8
CONFIRMED_FF_LO   = None   # e.g. 0.40
CONFIRMED_PMAX_LO = None   # e.g. 2.8
CONFIRMED_VOC_HI  = 0.747
CONFIRMED_ISC_HI  = None   # e.g. 5.80
CONFIRMED_FF_HI   = None   # e.g. 0.89
CONFIRMED_PMAX_HI = None   # e.g. 3.42

SPLIT_SEED = 42
SPLIT_TARGETS = {"train": 0.70, "val": 0.15, "test": 0.15}

print(f"ROOT     : {ROOT}")
print(f"PROCESSED: {PROCESSED}")
print(f"IV_JSON  : {IV_JSON}  exists={os.path.exists(IV_JSON)}")

ROOT     : /home/derk/DataScience/CAS_projects/LUCIA
PROCESSED: /home/derk/DataScience/CAS_projects/LUCIA/data/processed
IV_JSON  : /home/derk/DataScience/CAS_projects/LUCIA/cell_data_dictionary_backup.json  exists=True


In [70]:
# ── Cell 1: Load IV from JSON → DataFrame ───────────────────────────────────
# Purpose : build a tidy DataFrame keyed by canonical_name; surface duplicate keys
# Inputs  : IV_JSON
# Outputs : df_raw (all rows), collision_report (printed)

with open(IV_JSON) as f:
    raw_dict = json.load(f)

print(f"JSON keys: {len(raw_dict):,}")

# Map raw JSON keys → canonical_name; detect collisions
canon_to_raw = {}   # canonical_name -> list of original keys
for raw_key in raw_dict:
    c = canonical_name(raw_key)
    canon_to_raw.setdefault(c, []).append(raw_key)

collisions = {c: keys for c, keys in canon_to_raw.items() if len(keys) > 1}
print(f"Canonical names: {len(canon_to_raw):,}  |  key collisions: {len(collisions)}")
if collisions:
    print("\n── Key collisions (same canonical name, multiple JSON keys) ──")
    for c, keys in collisions.items():
        print(f"  {c}: {keys}")

# Build DataFrame — one row per canonical name (keep last on collision)
records = []
for c, keys in canon_to_raw.items():
    rec = dict(raw_dict[keys[-1]])   # last wins; collisions flagged above
    rec["cell_name"] = c
    rec["_collision"] = len(keys) > 1
    records.append(rec)

df_raw = pd.DataFrame(records).set_index("cell_name")
print(f"\ndf_raw shape: {df_raw.shape}")
print(df_raw.dtypes.to_string())

JSON keys: 11,203


Canonical names: 11,203  |  key collisions: 0



df_raw shape: (11203, 78)
id                         int64
time_x                    object
Pmax                     float64
Rs                       float64
Rsh                      float64
Vbreakdown               float64
TempDUT                  float64
Voc                      float64
Vpmax                    float64
Eff                      float64
FF                       float64
Gavg                     float64
InstName                  object
Isc                      float64
measurement_number         int64
measurement_letter        object
calc_area                float64
jsc                      float64
wafer_id                   int64
wafer_piece_id             int64
Rs_area                  float64
Rsh_area                 float64
el_id                    float64
spot_iv_id               float64
el_source_id             float64
pl_source_id             float64
time_y                    object
waferpiece_id            float64
spot_string               object
el_hi_avg_whole 

In [71]:
# ── Cell 2: Coerce numeric; NaN audit ───────────────────────────────────────
# Purpose : ensure IV columns are float64; report missingness before any cuts
# Inputs  : df_raw
# Outputs : df (typed), NaN count table

# Expected IV parameter column names — adjust if JSON uses different keys
IV_COLS = ["Voc", "Isc", "FF", "Pmax", "Vpmax", "Eff", "Rsh", "Rs", "Gavg", "Vbreakdown", "Isc", "jsc", "TempDUT"]

df = df_raw.copy()
for col in IV_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

present = [c for c in IV_COLS if c in df.columns]
missing = [c for c in IV_COLS if c not in df.columns]
if missing:
    print(f"WARNING — expected IV cols not found: {missing}")

nan_counts = df[present].isna().sum()
print("NaN counts per IV param:")
print(nan_counts[nan_counts > 0].to_string() if nan_counts.any() else "  none")
print(f"\nTotal rows: {len(df):,}")

NaN counts per IV param:
jsc    217

Total rows: 11,203


In [72]:
# ── Cell 3: Hard exclusions → iv_keep ───────────────────────────────────────
# Purpose : flag physically implausible cells; these are never reconsidered
# Inputs  : df, HARD thresholds
# Outputs : df with iv_keep column; exclusion breakdown printed

hard_fail = (
    (df["FF"]  < HARD["ff_lo"]) |
    (df["FF"]  > HARD["ff_hi"]) |
    (df["Voc"] < HARD["voc_lo"]) |
    (df["Isc"] < HARD["isc_lo"])
)

df["iv_keep"] = ~hard_fail

n_total   = len(df)
n_hard    = hard_fail.sum()
n_kept    = df["iv_keep"].sum()
print(f"Total:         {n_total:>6,}")
print(f"Hard excluded: {n_hard:>6,}  ({100*n_hard/n_total:.1f}%)")
print(f"Remaining:     {n_kept:>6,}")

# Breakdown by condition
for label, mask in [
    (f"FF < {HARD['ff_lo']}",   df["FF"]  < HARD["ff_lo"]),
    (f"FF > {HARD['ff_hi']}",   df["FF"]  > HARD["ff_hi"]),
    (f"Voc < {HARD['voc_lo']}", df["Voc"] < HARD["voc_lo"]),
    (f"Isc < {HARD['isc_lo']}", df["Isc"] < HARD["isc_lo"]),
]:
    print(f"  {label}: {mask.sum()}")

Total:         11,203
Hard excluded:    381  (3.4%)
Remaining:     10,822
  FF < 0.25: 335
  FF > 0.9: 1
  Voc < 0.5: 271
  Isc < 3.0: 241


In [73]:
# ── Cell 3b: Exclusion flags — shadow_masked & manual_excluded ───────────────
# Purpose : add two keep-flags to df from external lists (never hard-delete rows)
# Inputs  : masked_cells.feather  (72 shadow-mask cells from v21 signal criterion)
#           exclusion_lists*.json  (manual eye-picks, multiple dated files)
# Outputs : df['shadow_masked'], df['manual_excluded'] columns; overlap stats printed

import os as _os

MASKED_FEATHER = f"{ROOT}/masked_cells.feather"
EXCL_JSONS = [
    f"{ROOT}/exclusion_lists.json",
    f"{ROOT}/exclusion_lists_11_5_26.json",
    f"{ROOT}/exclusion_lists_12_6_26.json",
]

# ── Shadow-masked cells ──────────────────────────────────────────────────────
shadow_cells = set()
if _os.path.exists(MASKED_FEATHER):
    _mc = pd.read_feather(MASKED_FEATHER)
    shadow_cells = set(_mc.index.astype(str))
    print(f"masked_cells.feather : {len(shadow_cells)} shadow-masked cells loaded")
else:
    print(f"WARNING: {MASKED_FEATHER} not found — shadow_masked will be all False")

# ── Manual exclusions (union of all dated JSON files) ────────────────────────
manual_cells = set()
for _path in EXCL_JSONS:
    if _os.path.exists(_path):
        import json as _json
        with open(_path) as _f:
            _d = _json.load(_f)
        _batch = set(_d.get("manual_exclusions", []))
        manual_cells |= _batch
        print(f"{_os.path.basename(_path)}: {len(_batch)} manual exclusions  "
              f"(running union: {len(manual_cells)})")
    else:
        print(f"WARNING: {_path} not found")

# ── Apply flags ───────────────────────────────────────────────────────────────
df["shadow_masked"]   = df.index.isin(shadow_cells)
df["manual_excluded"] = df.index.isin(manual_cells)

n_shadow = df["shadow_masked"].sum()
n_manual = df["manual_excluded"].sum()
n_both   = (df["shadow_masked"] & df["manual_excluded"]).sum()
iv_keep_shadow = (df["shadow_masked"] & df["iv_keep"]).sum()
iv_keep_manual = (df["manual_excluded"] & df["iv_keep"]).sum()

print(f"\nSummary:")
print(f"  shadow_masked   : {n_shadow:>4}  (of which iv_keep=True: {iv_keep_shadow})")
print(f"  manual_excluded : {n_manual:>4}  (of which iv_keep=True: {iv_keep_manual})")
print(f"  both flags      : {n_both:>4}")
print(f"\nModelling cohort (iv_keep & ~shadow_masked & ~manual_excluded): "
      f"{(df['iv_keep'] & ~df['shadow_masked'] & ~df['manual_excluded']).sum():,} cells")

masked_cells.feather : 71 shadow-masked cells loaded
exclusion_lists.json: 33 manual exclusions  (running union: 33)
exclusion_lists_11_5_26.json: 19 manual exclusions  (running union: 33)
exclusion_lists_12_6_26.json: 17 manual exclusions  (running union: 49)

Summary:
  shadow_masked   :   57  (of which iv_keep=True: 57)
  manual_excluded :   46  (of which iv_keep=True: 45)
  both flags      :   16

Modelling cohort (iv_keep & ~shadow_masked & ~manual_excluded): 10,736 cells


In [74]:
# ── Cell 4: Candidate-cut count printout ────────────────────────────────────
# Purpose : show how many cells each threshold would add to exclusions so the
#           user can choose thresholds targeting ≈5 % total IV rejection
# Inputs  : df (after hard exclusions), CAND_* lists
# Outputs : table printed; NO changes to df yet
# !! STOP HERE — review output, fill CONFIRMED_* in the config cell, re-run Cell 5 !!

pool = df[df["iv_keep"]].copy()   # only cells that survived hard cuts
n_pool = len(pool)
print(f"Pool after hard cuts: {n_pool:,}")
print()

rows = []
for param, cands, direction in [
    ("Voc",  CAND_VOC_LO,  "lo"),
    ("Isc",  CAND_ISC_LO,  "lo"),
    ("FF",   CAND_FF_LO,   "lo"),
    ("Pmax", CAND_PMAX_LO, "lo"),
    ("Voc",  CAND_VOC_HI,  "hi"),
    ("Isc",  CAND_ISC_HI,  "hi"),
    ("FF",   CAND_FF_HI,   "hi"),
    ("Pmax", CAND_PMAX_HI, "hi"),
]:
    for thr in cands:
        if direction == "lo":
            n = (pool[param] < thr).sum()
            label = f"{param} < {thr}"
        else:
            n = (pool[param] > thr).sum()
            label = f"{param} > {thr}"
        rows.append({"cut": label, "n_excluded": n, "pct_pool": f"{100*n/n_pool:.2f}%"})

print(pd.DataFrame(rows).to_string(index=False))
print()
print("── Descriptive stats (pool) ──")
print(pool[[c for c in ["Voc","Isc","FF","Pmax"] if c in pool.columns]].describe().round(4).to_string())

Pool after hard cuts: 10,822

        cut  n_excluded pct_pool
 Voc < 0.68         399    3.69%
 Voc < 0.69         478    4.42%
  Voc < 0.7         580    5.36%
 Voc < 0.71         773    7.14%
  Isc < 4.8         219    2.02%
  Isc < 4.9         262    2.42%
  Isc < 5.0         316    2.92%
  Isc < 5.1         405    3.74%
   FF < 0.4         210    1.94%
  FF < 0.45         273    2.52%
   FF < 0.5         366    3.38%
 Pmax < 2.8        2080   19.22%
 Pmax < 2.9        2542   23.49%
 Pmax < 3.0        3066   28.33%
Voc > 0.747           1    0.01%
  Isc > 5.8           2    0.02%
  FF > 0.89           0    0.00%
Pmax > 3.42           1    0.01%

── Descriptive stats (pool) ──
              Voc         Isc          FF        Pmax
count  10822.0000  10822.0000  10822.0000  10822.0000
mean       0.7295      5.5117      0.7461      3.0119
std        0.0230      0.2204      0.0908      0.4418
min        0.5056      3.0306      0.2501      0.4623
25%        0.7309      5.5019      0.7426

In [75]:
# ── Cell 5: Apply confirmed candidate cuts ───────────────────────────────────
# Purpose : update iv_keep with the thresholds confirmed in the config cell
# Inputs  : df, CONFIRMED_* from config cell
# Outputs : df with updated iv_keep; rejection summary printed
# NOTE    : run AFTER filling in CONFIRMED_* above and re-running the config cell

confirmed_cuts = {
    "Voc_lo":  CONFIRMED_VOC_LO,
    "Isc_lo":  CONFIRMED_ISC_LO,
    "FF_lo":   CONFIRMED_FF_LO,
    "Pmax_lo": CONFIRMED_PMAX_LO,
    "Voc_hi":  CONFIRMED_VOC_HI,
    "Isc_hi":  CONFIRMED_ISC_HI,
    "FF_hi":   CONFIRMED_FF_HI,
    "Pmax_hi": CONFIRMED_PMAX_HI,
}

active = {k: v for k, v in confirmed_cuts.items() if v is not None}
if not active:
    print("No confirmed cuts set — iv_keep unchanged (only hard exclusions active).")
else:
    soft_fail = pd.Series(False, index=df.index)
    for key, thr in active.items():
        param, direction = key.rsplit("_", 1)
        if direction == "lo":
            soft_fail |= df[param] < thr
        else:
            soft_fail |= df[param] > thr

    df["iv_keep"] = df["iv_keep"] & ~soft_fail

    n_total   = len(df)
    n_kept    = df["iv_keep"].sum()
    n_removed = n_total - n_kept
    print(f"Active cuts:  {list(active.keys())}")
    print(f"Total:        {n_total:>6,}")
    print(f"Excluded:     {n_removed:>6,}  ({100*n_removed/n_total:.1f}%)")
    print(f"Kept:         {n_kept:>6,}")
    assert 100 * n_removed / n_total < 10, "WARNING: rejection >10% — check thresholds"

Active cuts:  ['Voc_hi']
Total:        11,203
Excluded:        382  (3.4%)
Kept:         10,821


In [ ]:
# ── Cell 5b: Compute and persist IV normalisation constants ────────────────
# Purpose : compute cohort max over the iv_keep set (hard+confirmed cuts applied,
#           shadow/manual flags NOT excluded — mirrors the full iv_keep definition).
#           Persisted to PROCESSED/iv_max_norm.json for use by all downstream NBs.
# Inputs  : df with iv_keep column (post-candidate-cuts)
# Outputs : IV_MAX dict + iv_max_norm.json

import json as _json

IV_TARGETS = ['Voc', 'Isc', 'Vmax', 'Imax', 'FF', 'Pmax', 'Rs']
iv_keep_df = df[df['iv_keep']]
IV_MAX = {tgt: float(iv_keep_df[tgt].max()) for tgt in IV_TARGETS}

_iv_max_path = os.path.join(PROCESSED, 'iv_max_norm.json')
_json.dump(IV_MAX, open(_iv_max_path, 'w'), indent=2)
print('IV_MAX saved:', {k: round(v, 4) for k, v in IV_MAX.items()})
print(f'Written to: {_iv_max_path}')


In [76]:
# ── Cell 6: Derive wo_group from cell_name ───────────────────────────────────
# Purpose : extract wafer-order identifier used to group cells for zero-leak splitting
# Inputs  : df index (canonical cell_name)
# Outputs : df['wo_group'] column

import re as _re

def extract_wo_group(cell_name: str) -> str:
    """'WO9999-7T2' -> 'WO9999'  |  'L1021-10S2' -> 'L1021'."""
    m = _re.match(r'((?:WO|L)\d+)', cell_name, _re.IGNORECASE)
    return m.group(1) if m else cell_name

df["wo_group"] = df.index.map(extract_wo_group)

wo_counts = df.loc[df["iv_keep"], "wo_group"].value_counts().sort_index()
print(f"WO groups in kept cells: {len(wo_counts)}")
print(wo_counts.to_string())

WO groups in kept cells: 165
wo_group
L1021       44
L1022        9
L1902       79
L1908      166
L1910       83
L1911       60
L1912       76
L1913       80
L1914       33
L1915       54
L1916       51
L1917       40
L1923       73
L1924       66
L1925       45
L1926       64
L1927       68
L1928       54
L1930       84
L1931       59
L1932       38
L1933       71
L1934       24
L1935       80
L1937       24
L1938        2
WO10000     64
WO10001     75
WO10002     69
WO10003     73
WO10004     80
WO10006     75
WO10007     28
WO10009     80
WO10010     81
WO10011     48
WO10013     28
WO10015     76
WO10016     76
WO10017    115
WO10018     50
WO10019     43
WO10020     47
WO10021     66
WO10022     49
WO10023     73
WO10024     59
WO10025     35
WO10026     31
WO10027     84
WO10028     72
WO10029     76
WO10030     32
WO10031     76
WO10032     71
WO10033     58
WO10034     77
WO10035     71
WO10036     50
WO10037     57
WO10040     64
WO10041     62
WO10042    126
WO10043     74
WO

In [77]:
# ── Cell 7: Random target-stratified 70/15/15 split ─────────────────────────
# Purpose : assign each kept cell to train/val/test; stratify on Pmax quantile
#           bins so target distributions match across splits; WOs freely span splits
# Inputs  : df (iv_keep, Pmax), SPLIT_SEED, SPLIT_TARGETS
# Outputs : df['split'] column

kept = df[df["iv_keep"]].copy()
kept['_pmax_bin'] = pd.qcut(kept['Pmax'].rank(method='first'), q=4, labels=False)

rng = np.random.default_rng(SPLIT_SEED)
split_col = pd.Series('train', index=kept.index)

for bin_val in kept['_pmax_bin'].unique():
    bin_idx = kept.index[kept['_pmax_bin'] == bin_val]
    n = len(bin_idx)
    n_test = max(1, round(n * SPLIT_TARGETS["test"]))
    n_val  = max(1, round(n * SPLIT_TARGETS["val"]))
    perm = rng.permutation(n)
    split_col[bin_idx[perm[:n_test]]]              = 'test'
    split_col[bin_idx[perm[n_test:n_test+n_val]]]  = 'val'

df['split'] = split_col

print("Split sizes (kept cells only):")
split_tbl = df[df["iv_keep"]].groupby("split").agg(
    n_cells=("split", "count"),
    n_wos=("wo_group", "nunique"),
)
split_tbl["pct"] = (split_tbl["n_cells"] / split_tbl["n_cells"].sum() * 100).round(1)
print(split_tbl.to_string())

# Sanity: WOs span multiple splits (no WO lock-in; this is deliberate)
wos_per_split = df[df["iv_keep"]].groupby("wo_group")["split"].nunique()
print(f"\nWOs spanning >1 split: {(wos_per_split > 1).sum()} / {len(wos_per_split)}")

Split sizes (kept cells only):
       n_cells  n_wos   pct
split                      
test      1624    163  15.0
train     7573    165  70.0
val       1624    163  15.0

WOs spanning >1 split: 163 / 165


In [78]:
# ── Cell 8: Save parquet + final report ─────────────────────────────────────
# Purpose : persist lucia_cells.parquet; print cohort summary for archiving
# Inputs  : df with all columns
# Outputs : OUT_PARQUET written to disk

# Keep only the schema columns (drop internal helpers like _collision)
iv_present = [c for c in ["Voc","Isc","FF","Pmax","Vmpp","Impp","Rsh","Rs","n","J0","Jph"]
              if c in df.columns]
out_cols = iv_present + ["wo_group", "split", "iv_keep",
                         "shadow_masked", "manual_excluded"]

# Guard: add columns with False if the exclusion-flags cell hasn't been run
for col in ("shadow_masked", "manual_excluded"):
    if col not in df.columns:
        df[col] = False

df_out = df[out_cols]

df_out.to_parquet(OUT_PARQUET, engine="pyarrow")
print(f"Saved: {OUT_PARQUET}  ({os.path.getsize(OUT_PARQUET)/1024:.0f} KB)")

print(f"\nTotal cells:    {len(df_out):,}")
print(f"iv_keep=True:   {df_out['iv_keep'].sum():,}")
print(f"shadow_masked:  {df_out['shadow_masked'].sum():,}")
print(f"manual_excluded:{df_out['manual_excluded'].sum():,}")
modelling = df_out["iv_keep"] & ~df_out["shadow_masked"] & ~df_out["manual_excluded"]
print(f"Modelling cohort (iv_keep & ~shadow & ~manual): {modelling.sum():,}")

print("\n── IV describe (kept cells) ──")
print(df_out.loc[df_out["iv_keep"], iv_present].describe().round(4).to_string())


Saved: /home/derk/DataScience/CAS_projects/LUCIA/data/processed/lucia_cells.parquet  (347 KB)

Total cells:    11,203
iv_keep=True:   10,821
shadow_masked:  57
manual_excluded:46
Modelling cohort (iv_keep & ~shadow & ~manual): 10,735

── IV describe (kept cells) ──
              Voc         Isc          FF        Pmax         Rsh          Rs
count  10821.0000  10821.0000  10821.0000  10821.0000  10821.0000  10821.0000
mean       0.7295      5.5117      0.7461      3.0121   1508.9190      0.0135
std        0.0230      0.2202      0.0907      0.4415   2325.2844      0.0263
min        0.5056      3.0306      0.2501      0.4623      0.0000      0.0008
25%        0.7309      5.5020      0.7427      2.9297     71.8595      0.0071
50%        0.7357      5.5710      0.7821      3.1837    567.0526      0.0079
75%        0.7384      5.6069      0.7948      3.2686   1836.8833      0.0096
max        0.7441      5.8433      0.8245      3.4427  21131.7124      0.5896
